# Tutorial 1 (Fixed)

Corrected version of `Tutorial1.ipynb`. Fixes applied:

1. **Task 1 metadata**: `N` = samples (`X.shape[0]`), `D` = features (`X.shape[1]`).
2. **Reproducibility**: noise columns are seeded (`np.random.seed`), Task 2 uses a seeded RNG.
3. **Task 1 timing**: warm-up excluded, `fit` and `predict` timed separately and averaged over repeats (the old first-row `1.4173 s` was first-call overhead).
4. **Consistent memory units**: Task 1 reports MB (train + test) with a matching label.
5. **Fair feature selection (Task 3)**: multiple variance thresholds and a random baseline matched to the reduced column counts.

Original `Tutorial1.ipynb` is left unchanged.

# Task 1

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

np.random.seed(42)  # make the injected noise reproducible
noise_cols = [0, 50, 200, 1000, 5000]
results = []

In [ ]:
print(f'Number of samples (N) = {X.shape[0]}')
print(f'Number of features (D) = {X.shape[1]}')
print('Training set shape (X, y):', X_train.shape, y_train.shape)
print('Testing set shape:', X_test.shape, y_test.shape)

In [ ]:
def benchmark_knn(X_tr, X_te, y_tr, y_te, n_neighbors=5, repeats=5):
    """Warm up, then time fit and predict separately; averages over `repeats`."""
    # Warm-up run (excluded from timing) avoids first-call / lazy-init overhead.
    warm = KNeighborsClassifier(n_neighbors=n_neighbors)
    warm.fit(X_tr, y_tr)
    warm.predict(X_te)

    fit_times, predict_times, accs = [], [], []
    for _ in range(repeats):
        knn = KNeighborsClassifier(n_neighbors=n_neighbors)

        t0 = time.perf_counter()
        knn.fit(X_tr, y_tr)
        fit_times.append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        preds = knn.predict(X_te)
        predict_times.append(time.perf_counter() - t0)

        accs.append(np.mean(preds == y_te) * 100.0)

    return np.mean(accs), np.mean(fit_times), np.mean(predict_times)

In [ ]:
for n_cols in noise_cols:
    if n_cols > 0:
        noise_train = np.random.uniform(0, 16, (X_train.shape[0], n_cols))
        noise_test = np.random.uniform(0, 16, (X_test.shape[0], n_cols))
        X_train_n = np.hstack((X_train, noise_train))
        X_test_n = np.hstack((X_test, noise_test))
    else:
        X_train_n, X_test_n = X_train.copy(), X_test.copy()

    acc, fit_t, pred_t = benchmark_knn(X_train_n, X_test_n, y_train, y_test)

    # Memory in MB, including both train and test data.
    mem_mb = (X_train_n.nbytes + X_test_n.nbytes) / 1024 / 1024

    results.append({
        'Noise columns': n_cols,
        'D': X_train_n.shape[1],
        'Accuracy (%)': round(acc, 2),
        'fit_time (s)': round(fit_t, 4),
        'predict_time (s)': round(pred_t, 4),
        'Memory (MB)': round(mem_mb, 3),
    })

In [ ]:
df = pd.DataFrame(results)
print(df.to_string(index=False))

plt.figure(figsize=(8, 5))
plt.plot(df['Noise columns'], df['Accuracy (%)'], marker='o')
plt.xlabel('Number of Noise Columns')
plt.ylabel('Task Accuracy (%)')
plt.title('Task 1: KNN Accuracy vs. Noise Columns')
plt.grid(True)
plt.show()

# Task 2

In [ ]:
from scipy.spatial.distance import cdist

In [ ]:
dimensions = [2, 10, 100, 1000, 10000]


def compute_contrast(N, dims, metric='euclidean', seed=42, n_queries=20):
    """Average distance contrast (d_max - d_min) / d_min over `n_queries` random queries.

    A fixed `seed` makes the result reproducible, and averaging over multiple
    query points removes single-sample noise from the estimate.
    """
    rng = np.random.default_rng(seed)
    contrasts = []
    for D in dims:
        points = rng.random((N, D))
        queries = rng.random((n_queries, D))

        distances = cdist(queries, points, metric=metric)
        d_min = distances.min(axis=1)
        d_max = distances.max(axis=1)
        contrasts.append(np.mean((d_max - d_min) / d_min))
    return contrasts

In [ ]:
contrast_500_euc = compute_contrast(500, dimensions, metric='euclidean', seed=42)
contrast_5000_euc = compute_contrast(5000, dimensions, metric='euclidean', seed=42)
contrast_500_man = compute_contrast(500, dimensions, metric='cityblock', seed=42)
contrast_5000_man = compute_contrast(5000, dimensions, metric='cityblock', seed=42)

print(f"{'D':>5} | {'N=500 (L2)':>12} | {'N=5000 (L2)':>13} | {'N=500 (L1)':>12} | {'N=5000 (L1)':>12}")
print('-' * 65)
for i, D in enumerate(dimensions):
    print(f"{D:>5} | {contrast_500_euc[i]:>12.4f} | {contrast_5000_euc[i]:>13.4f} | {contrast_500_man[i]:>12.4f} | {contrast_5000_man[i]:>12.4f}")

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(dimensions, contrast_500_euc, marker='o', label='N=500 (L2 - Euclidean)')
plt.plot(dimensions, contrast_5000_euc, marker='s', label='N=5000 (L2 - Euclidean)')
plt.plot(dimensions, contrast_500_man, marker='^', label='N=500 (L1 - Manhattan)')
plt.plot(dimensions, contrast_5000_man, marker='d', label='N=5000 (L1 - Manhattan)')

plt.xscale('log')
plt.xlabel('Dimensions (D) - Log Scale')
plt.ylabel('Contrast: (d_max - d_min) / d_min')
plt.title('Task 2: Distance Concentration')
plt.legend()
plt.grid(True)
plt.show()

# Task 3

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import VarianceThreshold

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


def evaluate_subset(X_tr, X_te, y_tr, y_te, seed=42):
    """Trains classifier and measures accuracy (%), training time (ms), and data memory (KB)."""
    clf = RandomForestClassifier(n_estimators=100, random_state=seed)

    start_time = time.perf_counter()
    clf.fit(X_tr, y_tr)
    train_time_ms = (time.perf_counter() - start_time) * 1000.0

    preds = clf.predict(X_te)
    acc_pct = accuracy_score(y_te, preds) * 100.0

    memory_kb = (X_tr.nbytes + X_te.nbytes) / 1024.0
    return acc_pct, train_time_ms, memory_kb


base_acc, base_time, base_mem = evaluate_subset(X_train, X_test, y_train, y_test)

In [ ]:
# Sweep several variance thresholds so reduced-dimension runs are comparable.
var_thresholds = [1.0, 5.0, 10.0, 22.0]
var_rows = []

for thresh in var_thresholds:
    selector = VarianceThreshold(threshold=thresh)
    X_train_var = selector.fit_transform(X_train)
    X_test_var = selector.transform(X_test)
    cols = X_train_var.shape[1]
    acc, t_ms, mem = evaluate_subset(X_train_var, X_test_var, y_train, y_test)
    var_rows.append({
        'Threshold': thresh,
        'Cols': cols,
        'Acc (%)': round(acc, 2),
        'Time (ms)': round(t_ms, 1),
        'Mem (KB)': round(mem, 1),
    })

var_df = pd.DataFrame(var_rows)
print(var_df.to_string(index=False))

In [ ]:
corr_matrix = pd.DataFrame(X_train).corr().abs().fillna(0)
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper_tri.columns if any(upper_tri[col] > 0.9)]

X_train_corr = np.delete(X_train, to_drop, axis=1)
X_test_corr = np.delete(X_test, to_drop, axis=1)
corr_cols = X_train_corr.shape[1]
corr_acc, corr_time, corr_mem = evaluate_subset(X_train_corr, X_test_corr, y_train, y_test)

In [ ]:
rng = np.random.RandomState(42)

# Match the random baseline to the reduced column counts (smallest variance set + correlation set).
random_sizes = sorted({int(var_df['Cols'].min()), int(corr_cols)})
rand_stats = {}

for size in random_sizes:
    accs, times, mems = [], [], []
    for i in range(10):
        cols = rng.choice(X_train.shape[1], size=size, replace=False)
        acc, t_ms, mem = evaluate_subset(X_train[:, cols], X_test[:, cols], y_train, y_test, seed=42 + i)
        accs.append(acc)
        times.append(t_ms)
        mems.append(mem)
    rand_stats[size] = (np.mean(accs), np.std(accs), np.mean(times), np.mean(mems))

In [ ]:
# Summary Table
print(f"Baseline (64 cols)                        : Acc = {base_acc:.2f}%                     | Time = {base_time:.1f} ms | Mem = {base_mem:.1f} KB")

best_var = var_df.loc[var_df['Acc (%)'].idxmax()]
print(f"Best variance (thresh={best_var['Threshold']}, {int(best_var['Cols'])} cols)    : Acc = {best_var['Acc (%)']:.2f}% ({best_var['Acc (%)'] - base_acc:+.2f} pp)          | Time = {best_var['Time (ms)']:.1f} ms | Mem = {best_var['Mem (KB)']:.1f} KB")

print(f"Correlation (>0.9, {corr_cols} cols)               : Acc = {corr_acc:.2f}% ({corr_acc - base_acc:+.2f} pp)          | Time = {corr_time:.1f} ms | Mem = {corr_mem:.1f} KB")

for size in random_sizes:
    m_acc, s_acc, m_time, m_mem = rand_stats[size]
    print(f"Random ({size} cols, 10x)                    : Acc = {m_acc:.2f}% ± {s_acc:.2f}% ({m_acc - base_acc:+.2f} pp) | Time = {m_time:.1f} ms | Mem = {m_mem:.1f} KB")